In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 94.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 4.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=30a4771dd5757a96f0861cb7e23a1f598d3621d541419e7a5f6f02b984f34399
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol without an attacker.

# HELPER: Quantum Random Number Generator

# randomness is produced by measuring qubits prepared in |+> = (1/sqrt(2))(|0> + |1>).  Measuring this state yields
# 0 or 1 with equal probability -- a true quantum coin flip.
simulator = BasicSimulator()

def quantum_random_bits(n):
    """Generate n random bits by measuring n qubits each prepared in |+>."""
    qc = QuantumCircuit(n, n)
    for i in range(n):
        qc.h(i)   # |0> --> |+> = (|0> + |1>) / sqrt(2)
    qc.measure(range(n), range(n))
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled, shots=1)
    counts = job.result().get_counts()
    bitstring = list(counts.keys())[0].replace(' ', '')
    # Qiskit stores results with qubit 0 as the rightmost character
    return [int(b) for b in reversed(bitstring)]



# ALICE: Choose random bits and bases, then encode each qubit

# Basis 0 = Z (rectilinear, +):  |0> and |1>
# Basis 1 = X (diagonal,    x):  |+> and |->

NUM_QUBITS  = 20

alice_bits  = quantum_random_bits(NUM_QUBITS)   # secret random bit string
alice_bases = quantum_random_bits(NUM_QUBITS)   # random choice of encoding basis

print('=== ALICE ===')
print(f'Bits:   {alice_bits}')
print(f'Bases:  {alice_bases}   (0 = Z/+,  1 = X/x)')

def alice_encode(bit, basis):
    """Return a 1-qubit circuit that encodes `bit` in `basis`.

    Z basis: bit=0 -> |0>,  bit=1 -> |1>   (standard computational states)
    X basis: bit=0 -> |+>,  bit=1 -> |->   (superposition states)
    """
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)     # flip to |1>
    if basis == 1:
        qc.h(0)     # rotate to diagonal basis
    return qc

# Each circuit acts as the qubit travelling through the quantum channel to Bob
encoded_qubits = [alice_encode(alice_bits[i], alice_bases[i])
                  for i in range(NUM_QUBITS)]

=== ALICE ===
Bits:   [0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0]
Bases:  [1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0]   (0 = Z/+,  1 = X/x)


In [3]:

# BOB: Choose random measurement bases and measure each qubit


bob_bases = quantum_random_bits(NUM_QUBITS)   # Bob's random measurement bases

def bob_measure(qc, basis):
    """Measure the qubit from circuit `qc` in `basis`.

    Measuring in X basis: apply H first to rotate back to Z, then measure.
    """
    qc_m = qc.copy()
    if basis == 1:
        qc_m.h(0)       # rotate from diagonal basis before measuring
    qc_m.measure(0, 0)
    compiled = transpile(qc_m, simulator)
    job = simulator.run(compiled, shots=1)
    counts = job.result().get_counts()
    return int(list(counts.keys())[0])

bob_results = [bob_measure(encoded_qubits[i], bob_bases[i])
               for i in range(NUM_QUBITS)]

print('=== BOB ===')
print(f'Bases:    {bob_bases}   (0 = Z/+,  1 = X/x)')
print(f'Results:  {bob_results}')

=== BOB ===
Bases:    [1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0]   (0 = Z/+,  1 = X/x)
Results:  [0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0]


In [4]:

# Alice and Bob publicly announce which bases they used

# They keep only the positions where both chose the same basis.
# The actual bit values are NOT revealed during sifting.

matching    = [i for i in range(NUM_QUBITS) if alice_bases[i] == bob_bases[i]]
alice_sifted = [alice_bits[i]  for i in matching]
bob_sifted   = [bob_results[i] for i in matching]

print('=== SIFTING ===')
print(f'Matching positions: {matching}')
print(f'Alice sifted key:   {alice_sifted}')
print(f'Bob   sifted key:   {bob_sifted}')
print(f'Sifted key length:  {len(alice_sifted)} bits  ',
      f'(expected ~{NUM_QUBITS // 2})')


# VERIFICATION: Sacrifice a sample to detect eavesdropping

# Alice and Bob publicly compare a random subset of their sifted keys.
# Any eavesdropper who measured qubits will have introduced errors.
# If the error rate exceeds the threshold they abort the protocol.

SAMPLE_SIZE = max(1, len(alice_sifted) // 4)   # reveal ~25% as check bits

sample_alice = alice_sifted[:SAMPLE_SIZE]
sample_bob   = bob_sifted[:SAMPLE_SIZE]
errors       = sum(a != b for a, b in zip(sample_alice, sample_bob))
error_rate   = errors / SAMPLE_SIZE

print()
print('=== VERIFICATION ===')
print(f'Check bits ({SAMPLE_SIZE}): Alice={sample_alice}  Bob={sample_bob}')
print(f'Errors: {errors}/{SAMPLE_SIZE}  -->  error rate = {error_rate:.1%}')

THRESHOLD = 0.25   # ~25% error rate expected when Eve intercepts every qubit

if error_rate > THRESHOLD:
    print(f'ATTACK DETECTED -- error rate {error_rate:.1%} exceeds '
          f'threshold {THRESHOLD:.1%}. Aborting.')
else:
    final_key = alice_sifted[SAMPLE_SIZE:]
    print(f'No attack detected -- error rate {error_rate:.1%} is within '
          f'threshold {THRESHOLD:.1%}.')
    print(f'Final shared key ({len(final_key)} bits): {final_key}')
    assert final_key == bob_sifted[SAMPLE_SIZE:], 'Key mismatch!'
    print('Key agreement confirmed.')

=== SIFTING ===
Matching positions: [0, 1, 2, 3, 4, 7, 12, 13, 14, 15, 17, 19]
Alice sifted key:   [0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0]
Bob   sifted key:   [0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0]
Sifted key length:  12 bits   (expected ~10)

=== VERIFICATION ===
Check bits (3): Alice=[0, 0, 0]  Bob=[0, 0, 0]
Errors: 0/3  -->  error rate = 0.0%
No attack detected -- error rate 0.0% is within threshold 25.0%.
Final shared key (9 bits): [1, 1, 0, 1, 1, 0, 0, 0, 0]
Key agreement confirmed.
